# Introduction
GOAL: To train a model to predict the wild fire risk of properties, using census data as input features and the proximity to fires as a target feature.

It would be cool to add in data regarding local climate.

In [ ]:
import pandas as pd
import load_wildfires
import load_census
import load_properties
import gis
import train
import visualize

from pathlib import Path
from sql_funcs import SQL

from settings import PATH_DATA

In [ ]:
sql_obj = SQL(test=True)


# Load Properties

## Select Properties of Interest

Chosing 300000 properties randomly from US addresses. We will join relevant census data to these addresses. This will probably take awhile, so best to run it overnight.

We don't care about the address itself. We add a census identifier called the GEOID which based on the coordinate's state, county, and tract number.

Using a package that makes use of the [US Census Geocoder API](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/census-geocoder.html), requests can be in batches of 10,000.

https://pypi.org/project/random-address/

In [ ]:
props = load_properties.Properties(sql_obj=sql_obj)
props.add_random_properties(100)


In [ ]:
properties = props.get_properties_gpd()

properties.tail()

# Load Features from US Census

2023 US Census Data

Using an API key, we will use the 'census' Python package to interact with the US Govermnent's census API.

In [ ]:
census = load_census.CensusData(sql_obj=sql_obj, year=2023, granularity='county')

In [ ]:
combined_gdf = census.merge_census_info(properties)
combined_gdf.head()


# Load Wildfire GIS Data for 2024

We will use point data from the Visible Infrared Imaging Radiometer Suite (VIIRS). A valid alternative is using burn boundary data. There are a few different data sources we could use, but in the interest of (portfolio) simplicity we'll use just the VIIRS.

N:B: May be a good chance to practice using AWS DB storage and retrieval?

In [ ]:
wildfires = load_wildfires.WildfireData(sql_obj=sql_obj)


In [ ]:
# Visualize wildfire locations
wildfire_map = wildfires.visualize_data(save_path=Path("figures/wildfires_map.html"))
wildfire_map

## Create Targets (Wildfire Proximity Score)

Give Each Property a Wildfire Risk Score based on the proximity to wildfires.

TODO: List the various options given for targets.


In [ ]:
proximity_features = gis.calc_all_features(combined_gdf, wildfires.data)
targets_features = pd.concat([combined_gdf, proximity_features], axis=1)

In [ ]:
targets_features.head()

In [ ]:
# Visualize properties colored by wildfire risk, with wildfire locations
combined_map = visualize.create_combined_map(
    targets_features, 
    wildfires.data, 
    risk_column="nearest_fire_km",
    save_path=Path("figures/risk_map.html")
)
combined_map

# Machine Learning Considerations
## Scoring Methods

For the float risk score, we can use Mean Squared Error (MSE) or Root Mean Squared Error (RMSE). Since it's quadratic in difference between observations and predictions deviations, MSE strongly penalizes large misses, which would be expensive for the insurance company.

For the risk category counts, they appear to be Poisson distributed, so a Poisson loss-function is appropriate.

For any classification model with the binned risk categories, we want to make large misses costly (i.e. predicting a 1 when the category is a 10), since these would also be very costly to the insurance company. To be honest, MSE will work here as well, since the categories are just 

# Model Machine Learning


NB: A good chance to make use of AWS compute.


### Split Data into Features/Targets

We use `nearest_fire_km` as the target — the distance in kilometres to the nearest wildfire detection. With the full 300k-property dataset, `exp_decay_score` (which captures both proximity and density of nearby fires) would be a better choice, but the small 119-property test set has nearly zero variance in decay score because all properties are ~360 km from the nearest fire.

All other proximity-derived columns are dropped so the model only sees census features as inputs.

In [ ]:
TARGET_COL = "nearest_fire_km"

# All proximity features are derived from the same wildfire data — drop them
# so the model only sees census features as inputs.
proximity_cols = [c for c in proximity_features.columns]
drop_cols = ["geometry", "geoid"] + [c for c in proximity_cols if c != TARGET_COL]

# SAVE TO PARQUET FOR AWS
# TODO: IS THIS REASONABLE TO DO HERE?
targets_features.drop(columns=drop_cols).to_parquet(PATH_DATA/"model_joined.parquet")

X_train, X_test, y_train, y_test = train.prepare_split(
    targets_features, TARGET_COL, drop_cols
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target range: {y_train.min():.4f} – {y_train.max():.4f}")
print(f"Target std:   {y_train.std():.6f}")
print(f"Target mean:  {y_train.mean():.4f}")

In [ ]:
# Diagnostic: check if the target has meaningful variance
cv = y_train.std() / y_train.mean() * 100  # coefficient of variation
print(f"Coefficient of variation: {cv:.4f}%")
if cv < 1.0:
    print(
        f"WARNING: Target has near-zero variance (CV={cv:.4f}%). "
        f"All properties are ~{y_train.mean():.1f} km from the nearest fire. "
        f"Models will appear to have perfect accuracy but are not learning meaningful patterns. "
        f"Scale to 300k properties for geographic diversity."
    )

### Preprocessing

Drop high-NaN columns, remove correlated features, then impute + scale. All steps are fit on training data only to prevent leakage.

In [ ]:
print(f"Features before filtering: {X_train.shape[1]}")

# Drop columns with >45% NaN (computed on train)
X_train, X_test = train.drop_high_nan_columns(X_train, X_test, threshold=0.45)
print(f"After NaN filter: {X_train.shape[1]}")

# Drop one of each highly correlated pair (|r| > 0.85)
X_train, X_test = train.drop_correlated_features(X_train, X_test, threshold=0.85)
print(f"After correlation filter: {X_train.shape[1]}")

# Save column names before pipeline converts to ndarray
feature_names = X_train.columns.tolist()

# Impute (KNN, k=5) then scale (StandardScaler)
pipeline = train.build_preprocessing_pipeline(n_neighbors=5)
X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

print(f"Final feature matrix: {X_train.shape}")

#### RandomForestRegressor


In [ ]:
rfr_search = train.train_random_forest(X_train, y_train, n_iter=20, cv=5)
print(f"Best RF params: {rfr_search.best_params_}")

rfr_metrics = train.evaluate_model(rfr_search.best_estimator_, X_train, X_test, y_train, y_test)
print(f"RF Train RMSE: {rfr_metrics['train_rmse']:.8f}")
print(f"RF Test  RMSE: {rfr_metrics['test_rmse']:.8f}")

#### XGBoost


In [ ]:
xgb_search = train.train_xgboost(X_train, y_train, n_iter=20, cv=5)
print(f"Best XGB params: {xgb_search.best_params_}")

xgb_metrics = train.evaluate_model(xgb_search.best_estimator_, X_train, X_test, y_train, y_test)
print(f"XGB Train RMSE: {xgb_metrics['train_rmse']:.8f}")
print(f"XGB Test  RMSE: {xgb_metrics['test_rmse']:.8f}")

#### Extract Feature Weights





In [ ]:
# Pick the better model
if xgb_metrics["test_rmse"] <= rfr_metrics["test_rmse"]:
    best_model = xgb_search.best_estimator_
    print("Best model: XGBoost")
else:
    best_model = rfr_search.best_estimator_
    print("Best model: RandomForest")

top_features = train.extract_feature_importance(best_model, feature_names, top_n=10)
print(f"\nTop 10 features:\n{top_features}")

In [ ]:
# Feature importance bar chart
fig_importance = visualize.plot_feature_importance(
    top_features, 
    title="Top 10 Feature Importances",
    save_path=Path("figures/feature_importance.png")
)
fig_importance

In [ ]:
# Actual vs Predicted scatter plot
y_pred = best_model.predict(X_test)

fig_scatter = visualize.plot_actual_vs_predicted(
    y_test.values, 
    y_pred,
    title="Actual vs Predicted (Test Set)",
    xlabel="Actual Distance to Fire (km)",
    ylabel="Predicted Distance to Fire (km)",
    save_path=Path("figures/actual_vs_predicted.png")
)
fig_scatter

In [ ]:
model_path = Path("Models") / "best_model.pkl"
train.save_model(best_model, model_path, pipeline=pipeline, feature_names=feature_names)
print(f"Model saved to {model_path}")

# Conclusion